# Scorecard — return and risk for every strategy, on one basis

**Headline strategies:** v1w, v2w and v3w. These are v1–v3 rebuilt so that each decides **once a week**: after the week's last close, it predicts Buy, Hold or Sell for the next 5 trading days, trades at the next open, and holds all week. See [docs/weekly_rebuild_note.md](../docs/weekly_rebuild_note.md). **Comparison:** the original v1–v3, which acted on a fresh prediction every trading day.

**One basis for all:**
- $100,000 start; 0.10% fee + 0.05% slippage per side;
- **cash earns the 13-week T-bill rate** (v1 as built assumed 0%), and **Sharpe is measured over T-bills**.

**Metrics:** final value, annual return (CAGR), annual volatility, maximum drawdown, Sharpe, Calmar (CAGR ÷ |max drawdown|), round trips, trading costs, time invested, and the share of 1,000 random-timing paths beaten (same time invested, same number of switches). The last is not applicable when a strategy switches at most once.

Every model is used exactly as frozen; nothing is selected here.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from src import config, v2, v3, weekly
from src.backtest import apply_rule, to_positions
from src.data import load_snapshot
from src.dataset import build_dataset_v2

O = config.OUTPUT_DIR
raw = load_snapshot(); px = raw[["Open", "Close"]]; ds = build_dataset_v2(raw)
rf = v2.load_tbill_daily(raw.index)
HOLD = config.ROOT / "data" / "holdout"
raw_h = pd.read_csv(HOLD / "spy_20261013.csv", index_col=0, parse_dates=True); px_h = raw_h[["Open", "Close"]]
rf_h = v2.load_tbill_daily(raw_h.index, HOLD / "irx_20261013.csv")
F1W = json.loads((O / "weekly" / "v1" / "frozen_config.json").read_text())
F2W = json.loads((O / "weekly" / "v2" / "frozen_config.json").read_text())
F3W = json.loads((O / "weekly" / "v3" / "frozen_config.json").read_text())
F3 = json.loads((O / "v3" / "frozen_config.json").read_text())
HGB = v3.v2_params()

ROWS, PATHS = [], {}
def score(period, strategy, schedule, held, prices, rfs, random_n=1000):
    rfs = rfs.loc[held.index]
    sim = v2.simulate(held, prices.loc[held.index], rfs)
    s = v2.summarize(sim, held, rfs, "")
    pct = None
    if s["round_trips"] > 1 and random_n:
        rand = v2.random_timing_benchmark(held, prices.loc[held.index], rfs, n=random_n)
        pct = round(float((rand < sim["value"].iloc[-1]).mean()), 3)
    ROWS.append({"period": period, "strategy": strategy, "schedule": schedule,
                 "final_value": s["final_value"], "cagr": s["cagr"], "ann_vol": s["ann_vol"],
                 "max_drawdown": s["max_drawdown"], "sharpe": s["sharpe_excess"],
                 "calmar": round(s["cagr"] / abs(s["max_drawdown"]), 2) if s["max_drawdown"] < 0 else None,
                 "round_trips": s["round_trips"], "costs": s["total_fees"], "time_invested": s["exposure"], "beats_random": pct})
    PATHS[(period, strategy)] = sim["value"]

def er_rule(er, default, sched):
    pos = weekly.weekly_positions_from_er(er, default) if sched == "weekly" else v2.positions_from_er(er, default)
    return v2.held_from_positions(pos, start=1 if default == "long" else 0)

## 2023–2024 test

In [2]:
P = "2023–24 test"
p1 = pd.read_csv(O / "test_predictions.csv", index_col=0, parse_dates=True)
proba1 = p1[["p_Sell", "p_Hold", "p_Buy"]].set_axis([-1, 0, 1], axis=1)
score(P, "Buy & Hold", "—", v2.buy_and_hold_held(p1.index), px, rf, 0)
m1w = None if F1W["margin"] == "argmax" else float(F1W["margin"])
score(P, "v1w", "weekly", v2.held_from_positions(weekly.weekly_positions_from_actions(apply_rule(proba1, m1w), start=0)), px, rf)
er2w = pd.read_csv(O / "weekly" / "v2" / "test_predictions.csv", index_col=0, parse_dates=True)["E_r"]
score(P, "v2w", "weekly", er_rule(er2w, F2W["default"], "weekly"), px, rf)
er3w = v3.expected_return(v3.walk_forward_proba(ds, F3W["treatment"], [2023, 2024], F3W["params"], F3W["model"]))
score(P, "v3w", "weekly", er_rule(er3w, F3W["default"], "weekly"), px, rf)
# original daily versions, for comparison
score(P, "v1", "daily", v2.held_from_positions(to_positions(p1["action"], min_hold=5)), px, rf)
er2 = pd.read_csv(O / "v2" / "test_predictions.csv", index_col=0, parse_dates=True)["E_r"]
score(P, "v2", "daily", er_rule(er2, "long", "daily"), px, rf)
er3 = v3.expected_return(v3.walk_forward_proba(ds, F3["treatment"], [2023, 2024], HGB))
score(P, "v3", "daily", er_rule(er3, F3["default"], "daily"), px, rf)

## 2013–2024 walk-forward (yearly refits; reporting only)

In [3]:
P = "2013–24 walk-forward"
years = list(range(2013, 2025))
er_v2w = v3.expected_return(v3.walk_forward_proba(ds, "C0", years, F2W["params"], F2W["model"]))
score(P, "Buy & Hold", "—", v2.buy_and_hold_held(er_v2w.index), px, rf, 0)
score(P, "v2w", "weekly", er_rule(er_v2w, F2W["default"], "weekly"), px, rf, 200)
er_v3w = v3.expected_return(v3.walk_forward_proba(ds, F3W["treatment"], years, F3W["params"], F3W["model"]))
score(P, "v3w", "weekly", er_rule(er_v3w, F3W["default"], "weekly"), px, rf, 200)
er_v2 = v3.expected_return(v3.walk_forward_proba(ds, "C0", years, HGB))
score(P, "v2", "daily", er_rule(er_v2, "long", "daily"), px, rf, 200)
er_v3 = v3.expected_return(v3.walk_forward_proba(ds, F3["treatment"], years, HGB))
score(P, "v3", "daily", er_rule(er_v3, F3["default"], "daily"), px, rf, 200)

## 2025–26 holdout (v3's primary holdout; v1 and v2 were not planned for it)

In [4]:
P = "2025–26 holdout"
ds_h = build_dataset_v2(raw_h)
er_h3w = v3.expected_return(v3.walk_forward_proba(ds_h, F3W["treatment"], [2025, 2026], F3W["params"], F3W["model"]))
score(P, "Buy & Hold", "—", v2.buy_and_hold_held(er_h3w.index), px_h, rf_h, 0)
score(P, "v3w", "weekly", er_rule(er_h3w, F3W["default"], "weekly"), px_h, rf_h)
er_h3 = v3.expected_return(v3.walk_forward_proba(ds_h, F3["treatment"], [2025, 2026], HGB))
score(P, "v3", "daily", er_rule(er_h3, F3["default"], "daily"), px_h, rf_h)

## The scorecard

In [5]:
card = pd.DataFrame(ROWS)
fmt = {"final_value": "${:,.0f}", "cagr": "{:.1%}", "ann_vol": "{:.1%}", "max_drawdown": "{:.1%}", "sharpe": "{:.2f}",
       "calmar": "{:.2f}", "costs": "${:,.0f}", "time_invested": "{:.0%}", "beats_random": "{:.0%}"}
for period, g in card.groupby("period", sort=False):
    print(period); display(g.drop(columns="period").set_index(["strategy", "schedule"]).style.format(fmt, na_rep="—"))
(O / "scorecard.json").write_text(json.dumps(json.loads(card.to_json(orient="records")), indent=2))
T = "2023–24 test"
paths = {"dates": [d.strftime("%Y-%m-%d") for d in PATHS[(T, "Buy & Hold")].index],
         **{k: [round(float(x)) for x in PATHS[(T, k)]] for k in ["Buy & Hold", "v1w", "v2w", "v3w"]}}
(O / "scorecard_paths.json").write_text(json.dumps(paths))
print("saved outputs/scorecard.json and outputs/scorecard_paths.json")

2023–24 test


,,final_value,cagr,ann_vol,max_drawdown,sharpe,calmar,round_trips,costs,time_invested,beats_random
strategy,schedule,,,,,,,,,,
Buy & Hold,—,"$156,546",25.2%,12.8%,-10.0%,1.43,2.53,1,$100,100%,—
v1w,weekly,"$126,864",12.7%,9.8%,-9.7%,0.76,1.31,11,"$2,554",60%,27%
v2w,weekly,"$110,453",5.1%,0.0%,0.0%,-0.71,—,0,$0,0%,—
v3w,weekly,"$156,546",25.2%,12.8%,-10.0%,1.43,2.53,1,$100,100%,—
v1,daily,"$129,069",13.7%,9.8%,-9.4%,0.85,1.45,15,"$3,317",58%,46%
v2,daily,"$121,812",10.4%,9.0%,-10.0%,0.59,1.04,17,"$3,754",53%,26%
v3,daily,"$156,546",25.2%,12.8%,-10.0%,1.43,2.53,1,$100,100%,—


2013–24 walk-forward


,,final_value,cagr,ann_vol,max_drawdown,sharpe,calmar,round_trips,costs,time_invested,beats_random
strategy,schedule,,,,,,,,,,
Buy & Hold,—,"$499,271",14.4%,16.7%,-33.7%,0.80,0.43,1,$100,100%,—
v2w,weekly,"$140,013",2.9%,3.0%,-4.7%,0.46,0.61,8,"$1,833",2%,100%
v3w,weekly,"$499,271",14.4%,16.7%,-33.7%,0.80,0.43,1,$100,100%,—
v2,daily,"$179,977",5.0%,11.6%,-22.4%,0.35,0.23,112,"$27,257",60%,32%
v3,daily,"$499,271",14.4%,16.7%,-33.7%,0.80,0.43,1,$100,100%,—


2025–26 holdout


,,final_value,cagr,ann_vol,max_drawdown,sharpe,calmar,round_trips,costs,time_invested,beats_random
strategy,schedule,,,,,,,,,,
Buy & Hold,—,"$134,410",18.2%,16.9%,-18.8%,0.84,0.97,1,$100,100%,—
v3w,weekly,"$134,410",18.2%,16.9%,-18.8%,0.84,0.97,1,$100,100%,—
v3,daily,"$134,410",18.2%,16.9%,-18.8%,0.84,0.97,1,$100,100%,—


saved outputs/scorecard.json and outputs/scorecard_paths.json
